In [ ]:
#| default_exp quality

# quality

> what to ask, and which sources are worth reading


`search()` ranks matches. This module decides what is worth reading: drop mirrors, boost authority for the query's intent, and stop one publisher owning the page. Used by `research()` when `curated=True`.


In [ ]:
#| export
import math, re
from collections import Counter
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit
from fastcore.all import ifnone


## Domain identity

`registrable_domain` / `host` answer whose site is this? `MULTI_LABEL_SUFFIXES` covers the country registries where that answer changes (`bbc.co.uk`, `nsw.gov.au`).


In [ ]:
#| export
#: Three-label registry suffixes (`bbc.co.uk`, not `co.uk`). Approximate PSL for countries we care about.
MULTI_LABEL_SUFFIXES = frozenset({
    'ac.at', 'co.at', 'gv.at', 'or.at', 'priv.at',
    'com.au', 'net.au', 'org.au', 'edu.au', 'gov.au', 'asn.au', 'id.au',
    'com.br', 'com.cn', 'edu.cn', 'gov.cn', 'net.cn', 'org.cn',
    'com.hk', 'edu.hk', 'gov.hk', 'org.hk',
    'co.in', 'edu.in', 'firm.in', 'gen.in', 'gov.in', 'ind.in', 'net.in', 'org.in',
    'ac.jp', 'co.jp', 'ed.jp', 'go.jp', 'ne.jp', 'or.jp',
    'com.mx', 'com.my', 'edu.my',
    'ac.nz', 'co.nz', 'net.nz', 'org.nz', 'govt.nz',
    'com.sg', 'edu.sg', 'gov.sg',
    'com.tr', 'com.ua',
    'ac.uk', 'co.uk', 'gov.uk', 'ltd.uk', 'me.uk', 'net.uk', 'org.uk', 'plc.uk', 'sch.uk',
    'co.za', 'org.za', 'gov.za',
})

def host(url:str) -> str:
    "The lowercase hostname of `url`, without `www.` or a trailing dot. `''` when there isn't one."
    try: h = urlsplit(url if '//' in ifnone(url,'') else f'//{ifnone(url,"")}').hostname or ''
    except ValueError: return ''
    h = h.strip().rstrip('.').lower()
    return h[4:] if h.startswith('www.') else h

def registrable_domain(url:str) -> str:
    "Site behind `url` (`docs.python.org`→`python.org`, `bbc.co.uk`→`bbc.co.uk`). IP/single-label unchanged."
    h = host(url)
    if not h or re.fullmatch(r'[\d.]+|\[[0-9a-f:]+\]', h): return h
    labels = [x for x in h.split('.') if x]
    if len(labels) < 3: return '.'.join(labels)
    return '.'.join(labels[-3:] if '.'.join(labels[-2:]) in MULTI_LABEL_SUFFIXES else labels[-2:])


## `norm_url`

One page → one key: drop scheme, fragment, tracking params, `www.`, default port; sort query. Used as the dedup key everywhere below.


In [ ]:
#| export
#: Click/tracking params — drop so the same article from six shares is one key.
_TRACKING = re.compile(r'^(utm_|fbclid|gclid|dclid|msclkid|yclid|mkt_tok|mc_[ce]id|igshid|'
                       r'vero_id|oly_(anon|enc)_id|_ga$|ref$|source$)')

_DFLT_PORT = {'http': 80, 'https': 443, 'ftp': 21}

def norm_url(u:str) -> str:
    "Dedup key: no scheme/fragment/tracking/`www.`/default port/trailing slash; query sorted."
    try: s = urlsplit(u)
    except ValueError: return u
    try: port = s.port
    except ValueError: port = None
    h = (s.hostname or '').rstrip('.').lower()
    h = h[4:] if h.startswith('www.') else h
    if ':' in h: h = f'[{h}]'                                   # ipv6 literal
    if port is not None and port != _DFLT_PORT.get(s.scheme.lower()): h = f'{h}:{port}'
    qs = sorted((k, v) for k, v in parse_qsl(s.query, keep_blank_values=True) if not _TRACKING.match(k))
    return urlunsplit(('', h, s.path.rstrip('/') or '/', urlencode(qs), ''))


## Authority by intent

Rules are keyed by intent (`docs`, `policy`, `release`, …). Shape: `docs.` (label), `.gov.au` (suffix), `sec.gov` (domain). No bare `startswith`, that would let `evil.example` inherit boosts.


In [ ]:
#| export
#: Boost/demote domains per intent — authority is not global.
DOMAIN_RULES = {
    'docs': dict(
        boost=('docs.', 'developer.', 'devdocs.', 'api.', 'github.com', 'readthedocs.io',
               'readthedocs.org', 'rust-lang.org', 'python.org', 'mozilla.org', 'w3.org',
               'ietf.org', 'rfc-editor.org'),
        demote=('medium.com', 'dev.to', 'reddit.com', 'stackoverflow.com', 'youtube.com',
                'geeksforgeeks.org', 'w3schools.com', 'tutorialspoint.com'),
    ),
    'policy': dict(
        boost=('.gov', '.gov.au', '.gov.uk', '.govt.nz', '.gov.in', '.gov.sg', '.gov.za',
               '.gc.ca', '.europa.eu', '.int', '.mil', '.edu', '.ac.uk', '.edu.au',
               'legislation.gov.uk', 'abcb.gov.au', 'standards.org.au', 'standards.org',
               'iso.org', 'nist.gov', 'oecd.org', 'who.int', 'eur-lex.europa.eu'),
        demote=('scribd.com', 'researchgate.net', 'slideshare.net', 'academia.edu', 'medium.com',
                'youtube.com', 'quora.com', 'pinterest.com', 'hipages.com.au', 'oneflare.com.au',
                'airtasker.com', 'serviceseeking.com.au', 'checkatrade.com', 'houzz.com',
                'angi.com', 'thumbtack.com'),
    ),
    'release': dict(
        boost=('github.com', 'blog.', 'news.', 'anthropic.com', 'openai.com', 'blog.google',
               'ai.google.dev', 'ai.meta.com', 'nvidia.com', 'microsoft.com', 'apple.com',
               'mistral.ai', 'tailscale.com'),
        demote=('youtube.com', 'medium.com', 'reddit.com', 'linkedin.com', 'facebook.com'),
    ),
    'security': dict(
        boost=('nvd.nist.gov', 'cve.org', 'cve.mitre.org', 'github.com/advisories', 'security.',
               'cert.', 'kb.cert.org', 'openwall.com', '.gov'),
        demote=('youtube.com', 'medium.com', 'reddit.com', 'linkedin.com'),
    ),
    'community': dict(
        boost=('reddit.com', 'news.ycombinator.com', 'stackexchange.com', 'stackoverflow.com',
               'forum.', 'forums.', 'community.', 'discourse.'),
        demote=('pinterest.com', 'quora.com', 'answers.com'),
    ),
}

def domain_matches(domain:str, rule:str) -> bool:
    "Match `docs.` (label), `.gov.au` (suffix), or exact/subdomain — never bare startswith."
    if not domain or not rule: return False
    if rule.endswith('.'): return domain.startswith(rule)
    # `.gov.uk` has to match the registry itself as well as everything under it, or the rule
    # covers every UK government department except gov.uk.
    if rule.startswith('.'): return domain == rule[1:] or domain.endswith(rule)
    if '/' in rule: return domain == rule.split('/', 1)[0]
    return domain == rule or domain.endswith(f'.{rule}')

def blocked_matches(domain:str, rule:str) -> bool:
    "True if host is in the blocked list (exact or subdomain)."
    return bool(domain) and (domain == rule or domain.endswith(f'.{rule}'))


## `classify`

Cue table → intent, or `None` when cues disagree. Abstain > guess.


In [ ]:
#| export
#: Cue words per intent class. Matched on word boundaries against the lowercased query.
INTENT_CUES = {
    'docs':      ('docs', 'documentation', 'api reference', 'reference manual', 'man page',
                  'how do i use', 'syntax', 'parameters', 'sdk', 'changelog api'),
    'policy':    ('regulation', 'regulations', 'regulatory', 'compliance', 'legislation', 'statute',
                  'law', 'laws', 'legal', 'permit', 'permits', 'planning permission', 'building code',
                  'building codes', 'building regulations', 'standard', 'standards', 'certification',
                  'licence', 'license required', 'zoning', 'council approval', 'tax rules'),
    'release':   ('release notes', 'changelog', 'release date', 'latest version', 'new version',
                  'announced', 'announcement', 'ships with', 'now available', 'launch'),
    'security':  ('cve', 'vulnerability', 'vulnerabilities', 'advisory', 'exploit', 'patch',
                  'security fix', 'zero day', 'rce'),
    'community': ('reddit', 'forum', 'hacker news', 'opinions', 'experiences', 'anyone else',
                  'worth it', 'real world', 'in practice', 'complaints', 'reviews from'),
}

_cue_rx = None
def _cue_pats():
    "One compiled pattern per class, built once."
    global _cue_rx
    if _cue_rx is None:
        _cue_rx = {k: re.compile('|'.join(rf'(?<![a-z0-9]){re.escape(c)}(?![a-z0-9])' for c in cues))
                   for k, cues in INTENT_CUES.items()}
    return _cue_rx

def classify(q:str) -> str:
    "Intent class from cue table, or None when cues disagree/tie."
    ql = (q or '').lower()
    hits = {k: len(rx.findall(ql)) for k, rx in _cue_pats().items()}
    hits = {k: n for k, n in hits.items() if n}
    if not hits: return None
    top = max(hits.values())
    winners = [k for k, n in hits.items() if n == top]
    return winners[0] if len(winners) == 1 else None


## Mirrors & `site:`

`SPAM_MIRRORS` are removed. Queries with `site:` skip spam filter and diversity caps, the caller already chose the domain.


In [ ]:
#| export
#: Sites that republish other people's answers. Removed rather than demoted: they add nothing over the
#: original, and fused metasearch actively promotes them — being optimised for every engine at once is
#: what they are for, and RRF reads that as consensus.
SPAM_MIRRORS = (
    # Stack Overflow / Q&A scrapers
    'newbedev.com', 'stackoom.com', 'stackovergo.com', 'syntaxfix.com', 'copyprogramming.com',
    'devcodef1.com', 'exceptionshub.com', 'code-examples.net', 'i-harness.com',
    'fixmycodeerror.com', 'stacklesson.com', 'itecnote.com', 'codeprozone.com',
    # GitHub issue / readme mirrors
    'githubmemory.com', 'gitmemory.com', 'issueexplorer.com', 'bleepcoder.com', 'gitanswer.com',
    # Documentation mirrors
    'w3cub.com', 'devdoc.net',
)

_SITE_OP = re.compile(r'\bsite:([a-z0-9][a-z0-9.-]*)', re.I)

def site_domains(q:str, include=()) -> list:
    "Domains from `site:` ops in `q` plus any explicit `include` list."
    ds = [d.lower().rstrip('.') for d in _SITE_OP.findall(q or '')]
    ds += [str(d).strip().lower() for d in (include or []) if str(d).strip()]
    return sorted(set(ds))

def _url(h) -> str: return h.get('href') or h.get('url') or '' if isinstance(h, dict) else str(h)

def drop_spam(hits:list, blocked=(), allowed=()) -> tuple:
    "Remove hits from mirror domains. Returns `(kept, removed_domains)`; `allowed` rescues a domain."
    rules = tuple(SPAM_MIRRORS) + tuple(str(d).strip().lower() for d in blocked if str(d).strip())
    ok = tuple(str(d).strip().lower() for d in allowed if str(d).strip())
    kept, removed = [], []
    for h in hits:
        d = host(_url(h))
        if d and not any(blocked_matches(d, r) for r in ok) and any(blocked_matches(d, r) for r in rules):
            removed.append(d); continue
        kept.append(h)
    return kept, sorted(set(removed))


## `plan`

Split a multi-part question into searches. Original query is always first; abstains to `[q]` when there is no second question.


In [ ]:
#| export
#: Conversational lead-ins stripped before splitting.
LEAD_INS = (r"i (?:want|need|would like|wanna|have) to", r"i(?:'m| am) (?:looking|trying) to",
            r"can you (?:tell me|help me|find|look up)", r"(?:please )?(?:tell me|help me|find me)",
            r"how (?:do|can|would) i", r"what (?:do|should) i", r"i (?:want|need)",
            r"give me", r"looking for")

#: Filler phrases with no search signal.
FILLER = (r"what (?:is|are|was|were) (?:the |a |an )?", r"things to consider(?: are)?",
          r"i (?:want|need) to know", r"(?:on|about|for|with) (?:it|this|that|them|these)",
          r"tell me", r"please", r"kindly", r"etc\.?", r"and so on", r"as well", r"also",
          r"in general", r"or not")

#: Words that carry no topic. Kept small on purpose: a stopword list that eats domain terms is worse
#: than none at all.
_STOP = frozenset("""a an the and or but of in on at to for from by with without into over under
    my your our their its his her this that these those it they i we you is are was were be been
    am do does did have has had can could should would will shall may might must not no if then
    than as so such about very more most much many some any all each every other another""".split())

_LEAD_RX = re.compile(r'^\s*(?:' + '|'.join(LEAD_INS) + r')\s+', re.I)
_FILL_RX = re.compile(r'\b(?:' + '|'.join(FILLER) + r')\b', re.I)
_SPLIT_RX = re.compile(r'\s*(?:[,;?]|\band\b|\bplus\b|\balso\b|\bas well as\b)\s*', re.I)
_WORD_RX = re.compile(r"[^\W_]+(?:['-][^\W_]+)*", re.UNICODE)

def _content(s:str) -> list:
    "The words in `s` that carry topic, in order, deduped."
    out, seen = [], set()
    for w in _WORD_RX.findall(_FILL_RX.sub(' ', s or '').lower()):
        if w in _STOP or w in seen: continue
        seen.add(w); out.append(w)
    return out

def facets(q:str) -> list:
    "The clauses of `q` as content-word lists, lead-in stripped. `[]` when there is nothing to split."
    parts = [p for p in _SPLIT_RX.split(_LEAD_RX.sub('', q or '').strip()) if p.strip()]
    return [c for p in parts if (c := _content(p))] if len(parts) > 1 else []

def plan(q:str,                  # the question as asked
         max_queries:int=4,      # total searches including the original; 1 disables planning
         min_words:int=6,        # below this a query has no room for two questions
         rewrite=None,           # optional `f(q) -> list[str]`, e.g. a model. Overrides the heuristic.
        ) -> list:
    "Split multi-part `q` into searches; always `[q]` first. Abstains when no second question."
    q = (q or '').strip()
    if not q: return []
    if rewrite is not None:
        extra = [str(s).strip() for s in (rewrite(q) or []) if str(s).strip()]
        return _dedup([q, *extra])[:max(1, max_queries)]
    if max_queries < 2 or len(q.split()) < min_words or site_domains(q): return [q]
    cs = facets(q)
    if len(cs) < 2: return [q]
    topic, out = cs[0], [q]
    for c in cs[1:]:
        if set(c) <= set(topic): continue                  # says nothing the subject did not
        out.append(' '.join(topic + [w for w in c if w not in topic]))
    return _dedup(out)[:max_queries]

def _dedup(qs:list) -> list:
    out, seen = [], set()
    for s in qs:
        if (k := ' '.join(s.lower().split())) not in seen: seen.add(k); out.append(s)
    return out

def interleave(lists:list, key=None) -> list:
    "Round-robin merge of result lists; dedup by `key` (default norm_url)."
    key = key or (lambda h: norm_url(h.get('href') or h.get('url') or ''))
    out, seen = [], {}
    for row in zip(*(list(l) + [None] * (max(map(len, lists), default=0) - len(l)) for l in lists)) \
            if lists else []:
        for h in row:
            if h is None: continue
            k = key(h)
            if k in seen: seen[k].setdefault('queries', []).extend(
                x for x in (h.get('queries') or []) if x not in seen[k].get('queries', []))
            else:
                seen[k] = dict(h); out.append(seen[k])
    return out


## Rerank

`authority_rerank` and `cap_per_domain` are stable and lossless: nothing dropped, demoted items keep relative order.


In [ ]:
#| export
def authority_rerank(hits:list, cls:str) -> tuple:
    "Stable boost/demote by DOMAIN_RULES for `cls`; lossless."
    rules = DOMAIN_RULES.get(cls or '')
    if not hits or not rules:
        return hits, dict(intent=cls, applied=False, boosted=[], demoted=[])
    boosted, demoted, scored = [], [], []
    for i, h in enumerate(hits):
        d = host(_url(h))
        s = (len(hits) - i) * 0.01                       # keep the incoming order as the tiebreak
        if any(domain_matches(d, r) for r in rules['boost']):  s += 10.0; boosted.append(d)
        if any(domain_matches(d, r) for r in rules['demote']): s -= 6.0;  demoted.append(d)
        scored.append((s, i, h))
    out = [h for _, _, h in sorted(scored, key=lambda t: (-t[0], t[1]))]
    return out, dict(intent=cls, applied=[_url(h) for h in out] != [_url(h) for h in hits],
                     boosted=sorted(set(boosted)), demoted=sorted(set(demoted)),
                     top_before=host(_url(hits[0])), top_after=host(_url(out[0])))

def cap_per_domain(hits:list, max_per:int=2) -> tuple:
    "Stable per-domain soft cap: overflow keeps relative order in the tail."
    if max_per < 1 or len(hits) < 3: return hits, 0
    head, tail, seen = [], [], Counter()
    for h in hits:
        d = registrable_domain(_url(h))
        if d and seen[d] >= max_per: tail.append(h); continue
        seen[d] += 1; head.append(h)
    return head + tail, len(tail)


## `cluster_sources`

Union-find over canonical URL + Jaccard text similarity. Counts independent stories, not URLs. Exact Jaccard (not MinHash), faster at research-scale `n`.


In [ ]:
#| export
def shingles(text:str, n:int=3) -> set:
    "Word n-grams of `text`. Fewer than `n` words has no evidence and yields nothing, never a match."
    w = _WORD.findall((text or '').casefold())
    return {tuple(w[i:i+n]) for i in range(len(w) - n + 1)} if len(w) >= n else set()

def jaccard(a:set, b:set) -> float:
    "|a ∩ b| / |a ∪ b|, and 0.0 when either side is empty — absence of evidence is not similarity."
    return len(a & b) / len(a | b) if a and b else 0.0

def _union_find(n:int):
    "Iterative find with path compression: a result page is small, but recursion here buys nothing."
    parent = list(range(n))
    def find(i):
        root = i
        while parent[root] != root: root = parent[root]
        while parent[i] != root: parent[i], i = root, parent[i]
        return root
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj: parent[max(ri, rj)] = min(ri, rj)      # keep the earliest index as the root
    return find, union

def cluster_sources(hits:list,             # search hits or research sources
                    threshold:float=0.6,   # trigram Jaccard above which two texts are one story
                   ) -> list:
    "Union-find clusters by canonical URL + Jaccard≥threshold; representative is first in input order."
    return _analyse(hits, threshold)[0]

def _analyse(hits:list, threshold:float) -> tuple:
    "Union-find by canonical URL + Jaccard≥threshold; first member is representative."
    hits = [h for h in hits if isinstance(h, dict)]
    n = len(hits)
    if not n: return [], 0
    find, union = _union_find(n)
    keys = [norm_url(_url(h)) for h in hits]
    sigs = [shingles(_text(h)) for h in hits]
    near = 0
    for i in range(n):
        for j in range(i):
            same = bool(keys[i]) and keys[i] == keys[j]
            alike = jaccard(sigs[i], sigs[j]) >= threshold
            near += alike
            if same or alike: union(i, j)
    groups = {}
    for i in range(n): groups.setdefault(find(i), []).append(i)
    clusters = [dict(canonical=_url(hits[g[0]]), title=hits[g[0]].get('title') or '',
                     members=[_url(hits[i]) for i in g], size=len(g),
                     domains=sorted({d for i in g if (d := registrable_domain(_url(hits[i])))}))
                for g in (groups[k] for k in sorted(groups))]
    return clusters, near

def independence(hits:list, threshold:float=0.6) -> dict:
    "Cluster count / n — fraction of independent stories."
    hits = [h for h in hits if isinstance(h, dict)]
    n = len(hits)
    if not n:
        return dict(n=0, sources=0, score=0.0, clusters=[], method='none', confidence='low',
                    limitations=[])
    cl = _analyse(hits, threshold)[0]
    textual = sum(1 for h in hits if shingles(_text(h))) >= 2
    fams = {d for c in cl for d in c['domains']}
    return dict(
        n=n, sources=len(cl), clusters=cl,
        score=round(0.7 * len(cl) / n + 0.3 * (len(fams) / n if fams else 0.0), 4),
        method='url+text' if textual else 'url',
        confidence='medium' if textual and n > 1 else 'low',
        limitations=['two sites reporting one press release in their own words are not detectable '
                     'by text overlap'] + ([] if textual else
                     ['too little text to compare: only identical urls were clustered']))


## `diversity`

Describe only: `dominant_domain`, near-dup pairs, independent `sources`. Never reorders.


In [ ]:
#| export
_WORD = re.compile(r'[^\W_]+', re.UNICODE)

def _trigrams(text:str) -> set:
    w = _WORD.findall((text or '').casefold())
    return {tuple(w[i:i+3]) for i in range(len(w) - 2)} if len(w) >= 3 else set()

def snippet_similarity(a:str, b:str) -> float:
    "Jaccard over word trigrams. Text shorter than three words scores 0 — no evidence, not a match."
    x, y = _trigrams(a), _trigrams(b)
    return len(x & y) / len(x | y) if x and y else 0.0

def _text(h) -> str:
    if not isinstance(h, dict): return ''
    return ' '.join(str(h.get(k) or '') for k in ('title', 'body', 'content', 'snippet', 'md'))

def diversity(hits:list, threshold:float=0.6) -> dict:
    "Describe dominant domain, near-dups, and independent source count."
    hits = [h for h in hits if isinstance(h, dict)]
    n = len(hits)
    if not n: return dict(score=0.0, n=0, domains=0, dominant_domain=None, dup_urls=0,
                          near_dups=0, sources=0)
    doms = Counter(d for d in (registrable_domain(_url(h)) for h in hits) if d)
    seen, dup_urls = set(), 0
    for h in hits:
        k = norm_url(_url(h))
        if k in seen: dup_urls += 1
        elif k: seen.add(k)
    clusters, near = _analyse(hits, threshold)
    pairs = n * (n - 1) // 2
    dom_div, url_uniq = (len(doms) / n), 1.0 - dup_urls / n
    content = 1.0 if not pairs else 1.0 - near / pairs
    top = doms.most_common(1)[0] if doms else None
    return dict(score=round(0.5*dom_div + 0.3*max(0., url_uniq) + 0.2*max(0., content), 4),
                n=n, domains=len(doms), dup_urls=dup_urls, near_dups=near,
                sources=len(clusters),
                dominant_domain=dict(domain=top[0], share=round(top[1]/n, 4)) if top else None)


## `curate`

drop spam → authority → cap → report. Returns the report even with `rerank=False`.


In [ ]:
#| export
def curate(q:str,                  # the query, read for intent and `site:` constraints
           hits:list,              # search hits (dicts with `href`/`url`, `title`, `body`)
           intent:str='auto',      # an intent class, 'auto' to classify, or None to skip authority
           max_per_domain:int=2,   # per-registrable-domain cap; 0 disables
           blocked=(), allowed=(), # extra mirror domains, and rescues from the built-in list
           rerank:bool=True,       # False: describe only, return `hits` untouched
          ) -> tuple:
    "Filter/rerank/describe hits → `(hits, report)`. `site:` skips spam drop and domain cap."
    sites = site_domains(q)
    rep = dict(intent=None, site_constrained=sites, dropped=[], demoted=0, authority=None)
    if not hits:
        return hits, dict(rep, diversity=diversity([]))
    out = hits
    if rerank:
        if not sites:
            out, rep['dropped'] = drop_spam(out, blocked=blocked, allowed=allowed)
        cls = classify(q) if intent == 'auto' else intent
        out, rep['authority'] = authority_rerank(out, cls)
        rep['intent'] = cls
        if max_per_domain and not sites:
            out, rep['demoted'] = cap_per_domain(out, max_per_domain)
    else:
        rep['intent'] = classify(q) if intent == 'auto' else intent
    rep['diversity'] = diversity(out)
    return out, rep


## Tests

In [ ]:
# registrable domain: the site behind a url, with the three-label registries handled
assert registrable_domain('https://docs.python.org/3/library/functions.html') == 'python.org'
assert registrable_domain('https://www.bbc.co.uk/news') == 'bbc.co.uk'
assert registrable_domain('https://planning.nsw.gov.au/x') == 'nsw.gov.au'
assert registrable_domain('https://wko.at/') == 'wko.at'
assert registrable_domain('https://x.com') == 'x.com' and registrable_domain('') == ''
assert registrable_domain('http://192.168.1.1/a') == '192.168.1.1'      # an ip has nothing to strip
assert host('https://WWW.Example.COM./p') == 'example.com'

In [ ]:
# norm_url: one page, one key -- and query order is not identity
assert norm_url('https://www.x.com/a/?utm_source=t#frag') == norm_url('http://x.com/a')
assert norm_url('https://x.com/a') != norm_url('https://x.com/b')
assert norm_url('https://x.com/a?b=2&a=1') == norm_url('https://x.com/a?a=1&b=2')   # sorted
assert norm_url('https://x.com:443/a') == norm_url('https://x.com/a')               # default port
assert norm_url('https://x.com:8443/a') != norm_url('https://x.com/a')              # but not any port
assert norm_url('https://x.com/a?mkt_tok=z&_ga=1&keep=2') == norm_url('https://x.com/a?keep=2')

In [ ]:
# rule matching is the security boundary, not just the ranking one
assert domain_matches('docs.python.org', 'docs.') and not domain_matches('notdocs.com', 'docs.')
assert domain_matches('planning.nsw.gov.au', '.gov.au')          # suffix rules cover a whole registry
assert domain_matches('gov.uk', '.gov.uk')                       # ...including the registry itself
assert domain_matches('abcb.gov.au', '.gov.au')
assert not domain_matches('hipages.com.au', '.gov.au')
assert domain_matches('sec.gov', 'sec.gov') and domain_matches('www.sec.gov', 'sec.gov')
assert not domain_matches('openai.com.evil.example', 'openai.com')   # never a bare startswith
assert blocked_matches('de.newbedev.com', 'newbedev.com')
assert not blocked_matches('newbedev.com.evil.example', 'newbedev.com')

In [ ]:
# intent: read it off the query, and abstain rather than guess
assert classify('rebuild my garage in australia, what is the regulation on it') == 'policy'
assert classify('planning permission for a garage') == 'policy'
assert classify('latest Tailscale release notes') == 'release'
assert classify('python asyncio api reference') == 'docs'
assert classify('CVE-2024-1234 advisory') is None or classify('log4j vulnerability advisory') == 'security'
assert classify('what do people on reddit think of it') == 'community'
assert classify('best turntables under 1000 euro') is None          # says nothing -> no reranking
assert classify('reddit release notes') is None                     # says two things -> no reranking
assert classify('') is None and classify(None) is None

In [ ]:
# spam mirrors are removed; an explicit site: constraint switches the whole thing off
_h = [dict(href='https://stackoverflow.com/questions/1', title='real'),
      dict(href='https://newbedev.com/questions/1', title='mirror'),
      dict(href='https://de.newbedev.com/q/1', title='mirror subdomain')]
_kept, _rm = drop_spam(_h)
assert [h['title'] for h in _kept] == ['real'] and _rm == ['de.newbedev.com', 'newbedev.com']
assert drop_spam(_h, allowed=['newbedev.com'])[0] == _h            # allow rescues the subdomain too

assert site_domains('site:reddit.com/r/LocalLLaMA best server') == ['reddit.com']
assert site_domains('plain query', include=['NSW.gov.au']) == ['nsw.gov.au']
_, _r = curate('site:reddit.com best local llm', [dict(href=f'https://reddit.com/{i}') for i in range(5)])
assert _r['site_constrained'] == ['reddit.com'] and _r['demoted'] == 0   # not diversified away

In [ ]:
# authority: the garage query, which is what this is all for
_garage = [dict(href='https://hipages.com.au/article/garage-cost', title='How much to build a garage'),
           dict(href='https://oneflare.com.au/costs/garage', title='Garage costs 2026'),
           dict(href='https://www.planning.nsw.gov.au/exempt-development', title='Exempt development'),
           dict(href='https://airtasker.com/garage', title='Get quotes'),
           dict(href='https://abcb.gov.au/ncc', title='National Construction Code')]
_out, _rep = curate('rebuild my garage in australia, what is the regulation on it', _garage)
assert _rep['intent'] == 'policy'
assert [host(_url(h)) for h in _out][:2] == ['planning.nsw.gov.au', 'abcb.gov.au'], [_url(h) for h in _out]
assert _rep['authority']['applied'] and _rep['authority']['top_before'] == 'hipages.com.au'

# nothing is dropped by reranking -- a lead-gen page is still a price signal, just not the answer
assert len(_out) == len(_garage)

# an unknown or absent intent is a no-op, not an error
assert authority_rerank(_garage, None)[0] == _garage
assert authority_rerank(_garage, 'nonsense')[1]['applied'] is False

In [ ]:
# capping and diversity describe the five-listicles-from-one-site failure
_mono = [dict(href=f'https://hipages.com.au/a/{i}', title='Garage cost guide', body='what a garage costs')
         for i in range(5)] + [dict(href='https://abcb.gov.au/ncc', title='NCC', body='construction code')]
_capped, _n = cap_per_domain(_mono, 2)
assert _n == 3 and [host(_url(h)) for h in _capped][:3] == ['hipages.com.au']*2 + ['abcb.gov.au']
assert len(_capped) == len(_mono)                                    # moved, never dropped

_d = diversity(_mono)
assert _d['n'] == 6 and _d['domains'] == 2
assert _d['dominant_domain'] == dict(domain='hipages.com.au', share=round(5/6, 4))
assert _d['near_dups'] > 0                                           # the five guides read alike
assert diversity([])['n'] == 0 and diversity([])['score'] == 0.0

# a genuinely diverse set scores higher than a monopolised one
_div = [dict(href=f'https://s{i}.com/a', title=f'topic {i}', body=f'quite different text number {i}')
        for i in range(6)]
assert diversity(_div)['score'] > _d['score']
assert snippet_similarity('one two three four', 'one two three four') == 1.0
assert snippet_similarity('too short', 'too short') == 0.0           # no trigram evidence -> not a match

In [ ]:
# curate is safe on the boring paths
assert curate('anything', [])[0] == []
_h2 = [dict(href='https://a.com/1', title='t')]
assert curate('q', _h2, rerank=False)[0] is _h2                      # describe-only changes nothing
assert 'diversity' in curate('q', _h2, rerank=False)[1]
assert curate('q', _h2, intent=None)[1]['intent'] is None

In [ ]:
# planning: one question containing three
_GARAGE = ('I want to rebuild my garage in australia, cheap options, '
           'what is the regulation on it, material options')
_p = plan(_GARAGE)
assert _p[0] == _GARAGE                                   # the baseline is never given up
assert len(_p) == 4                                       # all three facets, plus the original
assert any('regulation' in s for s in _p[1:]), _p
assert any('cheap' in s for s in _p[1:]) and any('material' in s for s in _p[1:]), _p
assert all('garage' in s and 'australia' in s for s in _p[1:]), _p   # subject re-attached to each facet
assert 'want' not in _p[1] and 'what' not in _p[1]        # lead-in and filler stripped

# ...and constant abstention, which is what makes it safe to leave on
assert plan('sqlite wal mode vs journal mode performance') == ['sqlite wal mode vs journal mode performance']
assert plan('garage cost') == ['garage cost']                        # too short to hold two questions
assert plan('site:reddit.com best local llm, cheap options') == ['site:reddit.com best local llm, cheap options']
assert plan(_GARAGE, max_queries=1) == [_GARAGE]                     # explicitly disabled
assert plan('') == []
assert len(plan(_GARAGE, max_queries=2)) == 2                        # the cap is honoured

# a facet that adds no words to the subject is not a search
assert plan('rebuild my garage in australia, the garage, australia') == \
       ['rebuild my garage in australia, the garage, australia']

# an injected rewriter (a model, in ramabana) replaces the heuristic and still keeps the original
assert plan('anything at all here', rewrite=lambda q: ['one', 'two']) == ['anything at all here', 'one', 'two']

In [ ]:
# interleaving is coverage, not consensus: the generalist that ranks for every facet must not
# outrank the specialist that owns one
_cost = [dict(href='https://hipages.com.au/guide', title='complete garage guide'),
         dict(href='https://oneflare.com.au/cost', title='costs')]
_reg  = [dict(href='https://hipages.com.au/guide', title='complete garage guide'),
         dict(href='https://planning.nsw.gov.au/exempt', title='exempt development')]
_mat  = [dict(href='https://hipages.com.au/guide', title='complete garage guide'),
         dict(href='https://bunnings.com.au/materials', title='materials')]
_il = interleave([_cost, _reg, _mat])
assert [host(_url(h)) for h in _il][:2] == ['hipages.com.au', 'oneflare.com.au']
assert 'planning.nsw.gov.au' in [host(_url(h)) for h in _il[:4]], [_url(h) for h in _il]
assert len(_il) == 4                                      # the guide appears once, not three times

# ragged lists and empties are fine
assert interleave([]) == [] and interleave([[], []]) == []
assert len(interleave([_cost, []])) == 2

In [ ]:
# clustering: four urls, one story -- and transitivity is what makes that visible
_wire = ('OpenAI said today that it is lowering prices for its newest model across all tiers, '
         'effective immediately for every API customer.')
_syn = [dict(href='https://a.invalid/x', title='OpenAI cuts prices', body=_wire),
        dict(href='https://b.invalid/y', title='OpenAI cuts prices', body=_wire),
        dict(href='https://c.invalid/z', title='OpenAI cuts prices', body=_wire),
        dict(href='https://openai.com/index/new-pricing', title='New pricing for our latest model',
             body='We are updating the per-token price of our newest model from today.')]
_cl = cluster_sources(_syn)
assert len(_cl) == 2, _cl                                  # three copies collapse, the original stands
assert _cl[0]['size'] == 3 and _cl[1]['size'] == 1
assert _cl[0]['canonical'] == 'https://a.invalid/x'        # named by its first (highest-ranked) member

_ind = independence(_syn)
assert _ind['n'] == 4 and _ind['sources'] == 2             # the number a reader actually needs
assert _ind['method'] == 'url+text' and _ind['confidence'] == 'medium'
assert _ind['limitations']                                 # always says what it cannot see

# transitivity: A~B and B~C is one family even where A and C alone fall short of the threshold
_chain = [dict(href='https://a.invalid/1', body='alpha beta gamma delta epsilon zeta eta theta'),
          dict(href='https://b.invalid/2', body='beta gamma delta epsilon zeta eta theta iota'),
          dict(href='https://c.invalid/3', body='gamma delta epsilon zeta eta theta iota kappa')]
assert jaccard(shingles(_chain[0]['body']), shingles(_chain[2]['body'])) < 0.6      # not a pair...
assert len(cluster_sources(_chain, threshold=0.5)) == 1, cluster_sources(_chain, 0.5)  # ...but one chain

# the same url twice is one source however differently it is described
assert len(cluster_sources([dict(href='https://x.com/a?utm_source=t', body='one two three four'),
                            dict(href='http://www.x.com/a', body='utterly different words here')])) == 1

# genuinely independent results stay independent
_div = [dict(href=f'https://s{i}.invalid/a', body=f'entirely separate subject number {i} here')
        for i in range(5)]
assert independence(_div)['sources'] == 5 and independence(_div)['score'] > independence(_syn)['score']

# no text to compare: degrade to url identity and say so rather than claiming confidence
_bare = [dict(href='https://a.invalid/1'), dict(href='https://b.invalid/2')]
assert independence(_bare)['method'] == 'url' and independence(_bare)['confidence'] == 'low'
assert any('too little text' in l for l in independence(_bare)['limitations'])
assert independence([]) == dict(n=0, sources=0, score=0.0, clusters=[], method='none',
                                confidence='low', limitations=[])

# and diversity carries the honest source count next to the url count
assert diversity(_syn)['n'] == 4 and diversity(_syn)['sources'] == 2
assert shingles('two words') == set() and jaccard(set(), {('a','b','c')}) == 0.0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()